In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
KNOWLEDGE_BASE_DIR = PROJECT_ROOT / "data" / "knowledge_base"

print("Knowledge base:", KNOWLEDGE_BASE_DIR)

Knowledge base: d:\ai-customer-support-assistant\data\knowledge_base


In [2]:
files = sorted(KNOWLEDGE_BASE_DIR.glob("*.md"))

documents = []

for file in files:
    documents.append({
        "source": file.name,
        "text": file.read_text(encoding="utf-8").strip()
    })

print("Documents loaded:", len(documents))

Documents loaded: 6


In [3]:
def chunk_text(text, chunk_size=500, chunk_overlap=100):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = end - chunk_overlap

    return chunks

In [4]:
test_chunks = chunk_text(documents[0]["text"])

print("Source:", documents[0]["source"])
print("Number of chunks:", len(test_chunks))

for i, chunk in enumerate(test_chunks):
    print(f"\n--- Chunk {i} ---")
    print(chunk)

Source: account.md
Number of chunks: 4

--- Chunk 0 ---
# Account Support

## Creating an Account

Customers can create an account by providing their email address and choosing a secure password. The email address should be valid and accessible because it may be required for account verification.

## Account Verification

If an account requires verification, customers should check their email for the verification message. They should also check their spam or junk folder if the message is not visible in the inbox.

## Forgot Password

Customers who fo

--- Chunk 1 ---
pam or junk folder if the message is not visible in the inbox.

## Forgot Password

Customers who forget their password should use the password reset option on the sign-in page. A password reset link will be sent to the registered email address.

## Password Reset Email Not Received

If the password reset email does not arrive, customers should check their spam or junk folder and confirm that they entered the correct email

In [5]:
all_chunks = []

for document in documents:
    chunks = chunk_text(document["text"])

    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "source": document["source"],
            "chunk_id": i,
            "text": chunk
        })

print("Documents:", len(documents))
print("Total chunks:", len(all_chunks))

Documents: 6
Total chunks: 27


In [6]:
chunks_df = pd.DataFrame(all_chunks)

chunks_df.head(10)

,source,chunk_id,text
0,account.md,0,# Account Support\n\n## Creating an Account\n\...
1,account.md,1,pam or junk folder if the message is not visib...
2,account.md,2,entered the correct email address. If the emai...
3,account.md,3,ccessful login attempts. Customers should wait...
4,billing.md,0,# Billing Support\n\n## Billing Cycle\n\nCusto...
5,billing.md,1,"ified, the customer should contact support wit..."
6,billing.md,2,their account billing section. If an invoice i...
7,billing.md,3,matically according to the customer's selected...
8,payments.md,0,# Payment Support\n\n## Payment Failed\n\nIf a...
9,payments.md,1,"of insufficient funds, incorrect payment infor..."


In [7]:
chunk_counts = (
    chunks_df
    .groupby("source")
    .size()
    .sort_values(ascending=False)
)

chunk_counts

source
technical_support.md    6
payments.md             5
billing.md              4
account.md              4
refunds.md              4
returns.md              4
dtype: int64

In [8]:
chunks_df["chunk_length"] = chunks_df["text"].str.len()

print("Minimum chunk length:", chunks_df["chunk_length"].min())
print("Maximum chunk length:", chunks_df["chunk_length"].max())
print("Average chunk length:", round(chunks_df["chunk_length"].mean(), 2))

Minimum chunk length: 217
Maximum chunk length: 500
Average chunk length: 473.22


In [9]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

chunks_path = PROCESSED_DIR / "chunks.csv"

chunks_df.to_csv(chunks_path, index=False)

print("Saved:", chunks_path)
print("Total chunks:", len(chunks_df))

Saved: d:\ai-customer-support-assistant\data\processed\chunks.csv
Total chunks: 27


In [10]:
loaded_chunks = pd.read_csv(chunks_path)

print(loaded_chunks.shape)
loaded_chunks.head()

(27, 4)


,source,chunk_id,text,chunk_length
0,account.md,0,# Account Support\n\n## Creating an Account\n\...,500
1,account.md,1,pam or junk folder if the message is not visib...,500
2,account.md,2,entered the correct email address. If the emai...,499
3,account.md,3,ccessful login attempts. Customers should wait...,347
4,billing.md,0,# Billing Support\n\n## Billing Cycle\n\nCusto...,499
